In [ ]:
import networkx as nx
import numpy as np
import pandas as pd
import random
from collections import defaultdict
import time

def load_graph(path):
    if path.endswith('.csv'):
        try:
            # Using sep=None and engine='python' forces Pandas to auto-detect delimiters
            df = pd.read_csv(path, sep=None, engine='python')
            if len(df.columns) < 2:
                raise ValueError("Not enough columns")
            G = nx.from_pandas_edgelist(df, source=df.columns[0], target=df.columns[1])
        except Exception:
            # Fallback for heavily malformed CSVs
            G = nx.read_edgelist(path, delimiter=',', comments='#')
    else:
        G = nx.read_edgelist(path, comments='#')
    G = nx.convert_node_labels_to_integers(G)
    return G

def generate_discrete_attributes(G, num_categories=3):
    return {node: random.randint(0, num_categories - 1) for node in G.nodes()}

def initialize_hybrid_representation(G):
    """
    Paper Section IV-A: Locus-based adjacency representation is used to get a
    relatively good initialization first, and then is decoded into the character
    string representation.
    """
    edges = []
    for node in G.nodes():
        neighbors = list(G.neighbors(node))
        if neighbors:
            edges.append((node, random.choice(neighbors)))

    # Decode to character string (node -> community ID)
    temp_G = nx.Graph()
    temp_G.add_nodes_from(G.nodes())
    temp_G.add_edges_from(edges)

    string_rep = {}
    for comm_id, component in enumerate(nx.connected_components(temp_G)):
        for node in component:
            string_rep[node] = comm_id

    return string_rep

def neighborhood_correction_strategy(X1, g, node_neighbors, Atr, attr_to_nodes):
    """Algorithm 1: Repairs genes based on attribute similarities"""
    neighbors = node_neighbors[g]
    l = len(neighbors)
    flag = 0
    n = 0

    if l != 0:
        shuffled_neighbors = list(neighbors)
        random.shuffle(shuffled_neighbors)

        for s in shuffled_neighbors:
            if X1[g] != X1[s] and Atr[g] == Atr[s]:
                X1[g] = X1[s]
                flag = 1
                n += 1
                if n >= (l / 3.0):
                    break

    if flag == 0 and l != 0:
        n = 0
        for s in shuffled_neighbors:
            if Atr[g] == Atr[s]:
                X1[g] = X1[s]
                flag = 1
                n += 1
                if n >= (l / 3.0):
                    break

    if flag == 0:
        if random.random() < 0.5 or l == 0:
            same_attr_nodes = attr_to_nodes.get(Atr[g], [])
            if same_attr_nodes:
                s = random.choice(same_attr_nodes)
                X1[g] = X1[s]
                flag = 1
        else:
            if l > 0:
                s = random.choice(neighbors)
                X1[g] = X1[s]
                flag = 1

    return X1

def hill_climbing_modularity(G, X1, node_neighbors, iterations=50):
    """
    Paper Section IV-C: Local search procedure. Attempts to find a better solution
    by incrementally changing a gene to one of its neighbors.
    """
    best_X = X1.copy()
    current_communities = get_communities_from_string_rep(best_X)
    best_mod = compute_modularity(G, current_communities)

    nodes = list(G.nodes())

    for _ in range(iterations):
        target = random.choice(nodes)
        neighbors = node_neighbors[target]
        if not neighbors: continue

        # Find neighbors in different communities
        diff_neighbors = [nbr for nbr in neighbors if best_X[target] != best_X[nbr]]
        if not diff_neighbors: continue

        candidate_nbr = random.choice(diff_neighbors)

        # Test the swap
        test_X = best_X.copy()
        test_X[target] = test_X[candidate_nbr]
        test_mod = compute_modularity(G, get_communities_from_string_rep(test_X))

        if test_mod > best_mod:
            best_mod = test_mod
            best_X = test_X

    return best_X, best_mod

def get_communities_from_string_rep(X):
    comm_dict = defaultdict(list)
    for node, comm_id in X.items():
        comm_dict[comm_id].append(node)
    return list(comm_dict.values())

def compute_modularity(G, communities):
    if len(communities) <= 1:
        return -1.0
    return nx.algorithms.community.quality.modularity(G, communities)

# --- Execution Block ---
datasets = [
    "/content/Dataset_CyberCrime_Sean.csv",
    "/content/london_crime_by_lsoa.csv",
    "/content/twitter_combined.txt.gz",
    "/content/facebook_combined (1).txt.gz",
    "/content/Meetings.csv",
    "/content/Phone_Calls.csv",
    "/content/Roles.csv",
    "/content/Sicilian Mafia.csv"
]

# Updated print formatting to include Nodes, Edges, and Communities
print(f"{'Dataset':<32} | {'Nodes':<6} | {'Edges':<7} | {'Comms':<6} | {'Modularity':<10} | {'Runtime (s)':<12}")
print("-" * 88)

for ds in datasets:
    try:
        start_time = time.time()

        G = load_graph(ds)

        # Extract Node and Edge counts
        num_nodes = G.number_of_nodes()
        num_edges = G.number_of_edges()

        Atr = generate_discrete_attributes(G, num_categories=3)

        node_neighbors = {n: list(G.neighbors(n)) for n in G.nodes()}
        attr_to_nodes = defaultdict(list)
        for node, attr_val in Atr.items():
            attr_to_nodes[attr_val].append(node)

        # 1. Initialize using Hybrid Representation
        X = initialize_hybrid_representation(G)

        # 2. Evolutionary Loop (Simplified for speed)
        generations = 5
        best_overall_modularity = -1.0

        for gen in range(generations):
            # Apply Algorithm 1 (Mutation/Repair)
            nodes_to_repair = random.sample(list(G.nodes()), min(50, G.number_of_nodes()))
            for g in nodes_to_repair:
                X = neighborhood_correction_strategy(X, g, node_neighbors, Atr, attr_to_nodes)

            # Apply Hill Climbing (Modularity Maximization)
            # Scales iterations based on graph size
            hc_iters = min(500, G.number_of_nodes())
            X, current_modularity = hill_climbing_modularity(G, X, node_neighbors, iterations=hc_iters)

            if current_modularity > best_overall_modularity:
                best_overall_modularity = current_modularity

        # Calculate final number of communities
        final_communities = get_communities_from_string_rep(X)
        num_communities = len(final_communities)

        runtime = time.time() - start_time
        ds_name = ds.split('/')[-1][:30]
        print(f"{ds_name:<32} | {num_nodes:<6} | {num_edges:<7} | {num_communities:<6} | {best_overall_modularity:<10.4f} | {runtime:<12.2f}")

    except FileNotFoundError:
        ds_name = ds.split('/')[-1][:30]
        print(f"{ds_name:<32} | {'-':<6} | {'-':<7} | {'-':<6} | {'File Not Found':<10} | {'N/A':<12}")
    except Exception as e:
        ds_name = ds.split('/')[-1][:30]
        print(f"{ds_name:<32} | {'-':<6} | {'-':<7} | {'-':<6} | {'Error':<10} | {str(e)[:12]:<12}")

Dataset                          | Nodes  | Edges   | Comms  | Modularity | Runtime (s) 
----------------------------------------------------------------------------------------
Dataset_CyberCrime_Sean.csv      | 136    | 161     | 14     | 0.5429     | 0.11        
london_crime_by_lsoa.csv         | 4868   | 4835    | 33     | 0.9645     | 184.11      
twitter_combined.txt.gz          | 81306  | 1342310 | 1782   | 0.2508     | 4013.32     
facebook_combined (1).txt.gz     | 4039   | 88234   | 98     | 0.5792     | 206.41      
Meetings.csv                     | 95     | 248     | 10     | 0.4816     | 0.23        
Phone_Calls.csv                  | 92     | 119     | 6      | 0.5249     | 0.09        
Roles.csv                        | 161    | 134     | 21     | 0.7522     | 0.07        
Sicilian Mafia.csv               | 143    | 325     | 11     | 0.4543     | 0.37        
